Cell 1

In [1]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import clone
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

Cell 2

In [2]:
# rebuild feature_df and groups
feature_df = pd.read_parquet("data/processed/feature_df.parquet").copy()
groups = feature_df["user"].astype(str)
y_all = feature_df["is_insider"].astype(int)

leak_cols = [
    "after_hours_events",
    "user_mean_after",
    "user_std_after",
    "after_hours_events_dev",
    "after_hours_ratio",
]
mean_std_cols = [c for c in feature_df.columns if c.startswith("user_mean_") or c.startswith("user_std_")]
drop_cols = list(set(leak_cols + mean_std_cols + ["user", "day", "employee_name", "email", "projects", "total_events"]))

X_all = feature_df.drop(columns=[c for c in drop_cols if c in feature_df.columns]).copy()
X_all = X_all.drop(columns=["is_insider"])

cat_cols = [c for c in ["role", "functional_unit", "department", "team", "supervisor"] if c in X_all.columns]
X_all = pd.get_dummies(X_all, columns=cat_cols, drop_first=False)

print("X_all:", X_all.shape)
print("Positives:", int(y_all.sum()))

X_all: (1394010, 434)
Positives: 25


Cell 3

In [3]:
def eval_probs(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    return {
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1],
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "pr_auc": average_precision_score(y_true, y_proba),
    }

Cell 4

In [4]:
def cross_val_model(name, model, X, y, groups, n_splits=5):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_rows = []
    
    for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X, y, groups), start=1):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
        
        m = clone(model)
        m.fit(X_tr, y_tr)
        proba = m.predict_proba(X_te)[:, 1]
        
        scores = eval_probs(y_te, proba, threshold=0.5)
        scores["model"] = name
        scores["fold"] = fold
        scores["positives_test"] = int(y_te.sum())
        scores["negatives_test"] = int((y_te == 0).sum())
        fold_rows.append(scores)
        
    return pd.DataFrame(fold_rows)

Cell 5

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


lr_model = LogisticRegression(class_weight="balanced", max_iter=5000)

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    min_child_weight=1,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    scale_pos_weight=10
)

cv_lr = cross_val_model("Logistic Regression", lr_model, X_all, y_all, groups, n_splits=5)
cv_rf = cross_val_model("Random Forest", rf_model, X_all, y_all, groups, n_splits=5)
cv_xgb = cross_val_model("XGBoost", xgb_model, X_all, y_all, groups, n_splits=5)

cv_results = pd.concat([cv_lr, cv_rf, cv_xgb], ignore_index=True)
cv_results

,tn,fp,fn,tp,precision,recall,f1,roc_auc,pr_auc,model,fold,positives_test,negatives_test
0,278796,0,0,1,1.000000,1.000000,1.000000,1.000000,1.000000,Logistic Regression,1,1,278796
1,278789,4,0,11,0.733333,1.000000,0.846154,0.999993,0.733333,Logistic Regression,2,11,278793
2,278798,1,0,1,0.500000,1.000000,0.666667,0.999998,0.500000,Logistic Regression,3,1,278799
3,278799,0,2,1,1.000000,0.333333,0.500000,0.651590,0.333346,Logistic Regression,4,3,278799
4,278798,0,0,9,1.000000,1.000000,1.000000,1.000000,1.000000,Logistic Regression,5,9,278798
5,278796,0,0,1,1.000000,1.000000,1.000000,1.000000,1.000000,Random Forest,1,1,278796
6,278789,4,0,11,0.733333,1.000000,0.846154,0.999988,0.662793,Random Forest,2,11,278793
7,278798,1,0,1,0.500000,1.000000,0.666667,1.000000,1.000000,Random Forest,3,1,278799
8,278799,0,1,2,1.000000,0.666667,0.800000,1.000000,1.000000,Random Forest,4,3,278799
9,278798,0,3,6,1.000000,0.666667,0.800000,1.000000,1.000000,Random Forest,5,9,278798


Cell 6

In [6]:
summary = cv_results.groupby("model").agg(
    folds=("fold", "count"),
    precision_mean=("precision", "mean"),
    precision_std=("precision", "std"),
    recall_mean=("recall", "mean"),
    recall_std=("recall", "std"),
    f1_mean=("f1", "mean"),
    f1_std=("f1", "std"),
    roc_auc_mean=("roc_auc", "mean"),
    roc_auc_std=("roc_auc", "std"),
    pr_auc_mean=("pr_auc", "mean"),
    pr_auc_std=("pr_auc", "std"),
    positives_test_mean=("positives_test", "mean"),
).reset_index()

summary

,model,folds,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,positives_test_mean
0,Logistic Regression,5,0.846667,0.225586,0.866667,0.298142,0.802564,0.217873,0.930316,0.155813,0.713336,0.297765,5.0
1,Random Forest,5,0.846667,0.225586,0.866667,0.182574,0.822564,0.119752,0.999998,0.000005,0.932559,0.150804,5.0
2,XGBoost,5,0.846667,0.225586,0.822222,0.255797,0.785641,0.152447,0.999996,0.000006,0.815603,0.253998,5.0


“The grouped cross-validation results confirm that the models generalize across users, with XGBoost and Logistic Regression outperforming Random Forest on recall and F1. The variability across folds indicates that the task remains challenging due to the rarity of insider cases.”

“After user-grouped cross-validation, XGBoost provided the best balance of recall and F1, with Logistic Regression performing comparably and Random Forest lagging behind. These results suggest that the engineered features contain strong predictive signal, although further validation on additional user splits would strengthen the conclusion.”

Cell 7

In [7]:
cv_results.to_csv("data/processed/cv_fold_results.csv", index=False)
summary.to_csv("data/processed/cv_summary.csv", index=False)

In [8]:
import joblib

joblib.dump(xgb_model, "data/models/xgb_cv.joblib")

['data/models/xgb_cv.joblib']